# The Adversarial Critic Loop

In [1]:
import os 
import asyncio
import yfinance as yf
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool
from pydantic import BaseModel, Field
from typing import Literal
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
@function_tool
def get_fundamentals(ticker: str) -> dict:
    """
    Fetches fundamental financial metrics.
    Args:
        ticker: Stock ticker symbol e.g. 'NVDA'
    """
    info = yf.Ticker(ticker).info
    return {
        "ticker": ticker,
        "current_price": info.get("currentPrice"),
        "pe_ratio": info.get("trailingPE"),
        "forward_pe": info.get("forwardPE"),
        "revenue_growth": info.get("revenueGrowth"),
        "profit_margins": info.get("profitMargins"),
        "return_on_equity": info.get("returnOnEquity"),
        "earnings_growth": info.get("earningsGrowth"),
    }

@function_tool
def get_risk_metrics(ticker: str) -> dict:
    """
    Fetches risk and volatility metrics.
    Args:
        ticker: Stock ticker symbol e.g. 'NVDA'
    """
    info = yf.Ticker(ticker).info
    return {
        "ticker": ticker,
        "beta": info.get("beta"),
        "debt_to_equity": info.get("debtToEquity"),
        "current_ratio": info.get("currentRatio"),
        "52_week_high": info.get("fiftyTwoWeekHigh"),
        "52_week_low": info.get("fiftyTwoWeekLow"),
        "short_ratio": info.get("shortRatio"),
    }

@function_tool
def get_market_sentiment(ticker: str) -> dict:
    """
    Fetches market sentiment indicators.
    Args:
        ticker: Stock ticker symbol e.g. 'NVDA'
    """
    info = yf.Ticker(ticker).info
    return {
        "ticker": ticker,
        "analyst_recommendation": info.get("recommendationKey"),
        "target_price": info.get("targetMeanPrice"),
        "number_of_analysts": info.get("numberOfAnalystOpinions"),
        "institutional_ownership": info.get("heldPercentInstitutions"),
        "insider_ownership": info.get("heldPercentInsiders"),
    }

In [3]:
class Challenge(BaseModel):
    claim: str
    reasoning: str
    severity: Literal["low","high"]

class CriticOutput(BaseModel):
    challenges: list[Challenge]
    blocks_publication: bool
    overall_assessment: str

In [4]:
# Specialist agents (same as Layer 5)
fundamentals_agent = Agent(
    name="FundamentalsAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a fundamental analysis specialist.
    ALWAYS call get_fundamentals first.
    Output 3 bullet points with specific numbers.
    End with: FUNDAMENTALS_SCORE: [1-10]
    """,
    tools=[get_fundamentals]
)

risk_agent = Agent(
    name="RiskAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a risk assessment specialist.
    ALWAYS call get_risk_metrics first.
    Output 3 bullet points with specific numbers.
    End with: RISK_SCORE: [1-10] (10 = highest risk)
    """,
    tools=[get_risk_metrics]
)

sentiment_agent = Agent(
    name="SentimentAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a market sentiment specialist.
    ALWAYS call get_market_sentiment first.
    Output 3 bullet points with specific numbers.
    End with: SENTIMENT_SCORE: [1-10]
    """,
    tools=[get_market_sentiment]
)

# Synthesizer
synthesizer_agent = Agent(
    name="SynthesizerAgent",
    model="gpt-4o-mini",
    instructions="""
    You are a senior investment analyst.
    Produce a structured investment report from specialist research.
    
    ## Investment Report: [TICKER]
    
    ### Key Findings
    - Fundamentals: [summary + score]
    - Risk: [summary + score]
    - Sentiment: [summary + score]
    
    ### Overall Score
    [Weighted: Fundamentals 40% + Sentiment 30% + Risk inverted 30%]
    
    ### Verdict
    [STRONG BUY / BUY / HOLD / AVOID]
    
    ### Reasoning
    [2-3 sentences using only data provided to you]
    
    ### Key Risk to Watch
    [Single biggest risk from the data]
    """
)

critic_agent = Agent(
    name="CriticAgent",
    model="gpt-4o-mini",
    instructions="""
    You are an adversarial reviewer for investment reports.
    Your job is to find claims that are NOT supported by the provided data.
    
    Check every claim in the report:
    - Is this number actually in the specialist data?
    - Is this conclusion logically supported by the data?
    - Did the synthesizer make any assumptions not in the data?
    
    Be strict. A STRONG BUY verdict needs strong evidence.
    If the data shows high risk, the verdict must acknowledge it.
    
    Output structured challenges only. No free-form commentary.
    """,
    output_type=CriticOutput      
)

In [9]:
async def run_research_pipeline(ticker:str)->str:
    print(f" [1/4] Running specialists... {ticker}")
    fundamentals_result, risk_result, sentiment_result = await asyncio.gather(
        Runner.run(fundamentals_agent, f"Analyze {ticker}"),
        Runner.run(risk_agent, f"Analyze {ticker}"),
        Runner.run(sentiment_agent, f"Analyze {ticker}")
    )
    combined_data = f"""
    Fundamentals:\n{fundamentals_result}\n
    Risk:\n{risk_result}\n
    Sentiment:\n{sentiment_result}
    """
    print(f" [2/4] Synthesizing report...")

    synthesis = await Runner.run(synthesizer_agent, combined_data)
    report_v1 = synthesis.final_output
    display(Markdown(report_v1))

    print(f" [3/4] Critiquing report...")
    critique = await Runner.run(critic_agent, report_v1)
    critic_output: CriticOutput = critique.final_output

    print(f"Challanges found: {len(critic_output.challenges)}")
    print(f"Blocks publication: {critic_output.blocks_publication}")
    print(f"Overall assessment: {critic_output.overall_assessment}")

    if critic_output.blocks_publication:
        print("[4/4] Revising report based on critique...")
        revision_instructions = f"""
        ORIGINAL RESEARCH DATA:
        {combined_data}

        YOUR PREVIOUS REPORT:
        {report_v1}

        CRITIQUE:
        {[c.model_dump() for c in critic_output.challenges]}

        Revise the report. For each challenge either:
        - Correct the claim using only data provided
        - Remove the claim if it cannot be supported
        - Lower confidence if data is ambiguous
        """

        revision = await Runner.run(synthesizer_agent, revision_instructions)
        final_report = revision.final_output
        display(Markdown(final_report))
    else:
        print("[4/4] No critical issues found. Report is ready for publication.")
        final_report = report_v1
    
    return final_report


In [10]:
final = await run_research_pipeline("NVDA")
print("Final report ready for publication:")
display(Markdown(final))

 [1/4] Running specialists... NVDA
 [2/4] Synthesizing report...


## Investment Report: NVDA

### Key Findings
- **Fundamentals:** NVIDIA displays strong fundamentals with a current price of $208.64, a P/E ratio of 31.95, and impressive revenue growth of 85.2%. **Score: 8**
- **Risk:** The company has a high beta of 2.202 and a debt-to-equity ratio of 6.555, indicating significant risk. However, its current ratio of 3.441 shows solid liquidity. **Score: 8**
- **Sentiment:** Analysts recommend NVDA as a Strong Buy, with a target price of $298.42, and institutional ownership at 70.86% indicates strong confidence in the stock. **Score: 9**

### Overall Score
Using the weighted formula (Fundamentals 40% + Sentiment 30% + Risk inverted 30%):
- Overall Score = (8 * 0.4) + (9 * 0.3) + ((10 - 8) * 0.3) = 3.2 + 2.7 + 0.6 = **6.5**

### Verdict
**BUY**

### Reasoning
NVIDIA's robust revenue growth and strong analyst support indicate a healthy investment opportunity. The high volatility and significant leverage present cautionary aspects, yet solid liquidity mitigates some risks.

### Key Risk to Watch
The largest risk is the elevated debt-to-equity ratio of 6.555, which could lead to financial instability if market conditions worsen.

 [3/4] Critiquing report...
Challanges found: 5
Blocks publication: True
Overall assessment: The investment report lacks sufficient evidence and logical connections for key claims, particularly concerning risk factors and analyst recommendations.
[4/4] Revising report based on critique...


## Investment Report: NVDA

### Key Findings
- **Fundamentals:** NVIDIA shows a current price of $208.64, with a P/E ratio of 31.95 and revenue growth of 85.2%. While these metrics indicate strong performance, industry comparisons could provide additional context. **Score: 8**
- **Risk:** The company exhibits a high beta of 2.202, indicating considerable volatility compared to the market. The debt-to-equity ratio of 6.555 suggests significant leverage, raising financial stability concerns. However, a current ratio of 3.441 implies that NVIDIA has a solid ability to cover its short-term liabilities. **Score: 8**
- **Sentiment:** The consensus among analysts is a Strong Buy, with a target price of $298.42. Institutional ownership at 70.86% reflects robust confidence in NVIDIA, suggesting strong market sentiment. Specific analysis supporting this target could enhance reliability. **Score: 9**

### Overall Score
Using the weighted formula (Fundamentals 40% + Sentiment 30% + Risk inverted 30%):
- Overall Score = (8 * 0.4) + (9 * 0.3) + ((10 - 8) * 0.3) = 3.2 + 2.7 + 0.6 = **6.5**

### Verdict
**BUY**

### Reasoning
NVIDIA's high revenue growth and positive analyst sentiment suggest a promising investment opportunity. However, the company’s notable volatility and leverage present cautionary signs, though solid liquidity may address some immediate risk concerns.

### Key Risk to Watch
The primary risk to monitor is the high debt-to-equity ratio of 6.555, which could lead to financial instability under adverse market conditions.

Final report ready for publication:


## Investment Report: NVDA

### Key Findings
- **Fundamentals:** NVIDIA shows a current price of $208.64, with a P/E ratio of 31.95 and revenue growth of 85.2%. While these metrics indicate strong performance, industry comparisons could provide additional context. **Score: 8**
- **Risk:** The company exhibits a high beta of 2.202, indicating considerable volatility compared to the market. The debt-to-equity ratio of 6.555 suggests significant leverage, raising financial stability concerns. However, a current ratio of 3.441 implies that NVIDIA has a solid ability to cover its short-term liabilities. **Score: 8**
- **Sentiment:** The consensus among analysts is a Strong Buy, with a target price of $298.42. Institutional ownership at 70.86% reflects robust confidence in NVIDIA, suggesting strong market sentiment. Specific analysis supporting this target could enhance reliability. **Score: 9**

### Overall Score
Using the weighted formula (Fundamentals 40% + Sentiment 30% + Risk inverted 30%):
- Overall Score = (8 * 0.4) + (9 * 0.3) + ((10 - 8) * 0.3) = 3.2 + 2.7 + 0.6 = **6.5**

### Verdict
**BUY**

### Reasoning
NVIDIA's high revenue growth and positive analyst sentiment suggest a promising investment opportunity. However, the company’s notable volatility and leverage present cautionary signs, though solid liquidity may address some immediate risk concerns.

### Key Risk to Watch
The primary risk to monitor is the high debt-to-equity ratio of 6.555, which could lead to financial instability under adverse market conditions.